# Classificação de Doença Cardiovascular
**Dataset:** heart_disease_dataset.csv  
**Algoritmos:** KNN e Regressão Logística  
**Objetivo:** Classificar se um paciente tem ou não doença cardiovascular (variável `Heart Disease`)

## Importação das Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
print('Bibliotecas carregadas com sucesso!')

---
## 1. Carregamento e Exploração dos Dados

In [ ]:
df = pd.read_csv('heart_disease_dataset.csv')

print(f'Dimensões do dataset: {df.shape[0]} linhas x {df.shape[1]} colunas')
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Verificar valores nulos
print('Valores nulos por coluna:')
print(df.isnull().sum())

In [ ]:
# Distribuição da variável alvo
print('Distribuição da variável alvo (Heart Disease):')
print(df['Heart Disease'].value_counts())
print(f'\nProporção (%):')
print((df['Heart Disease'].value_counts(normalize=True) * 100).round(2))

---
## 2. Transformações e Pré-Processamento

In [ ]:
# Identificar colunas categóricas e numéricas
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols.remove('Heart Disease')

print(f'Colunas categóricas: {cat_cols}')
print(f'Colunas numéricas:   {num_cols}')

In [ ]:
# Visualização da distribuição da variável alvo
fig, ax = plt.subplots(figsize=(6, 4))
df['Heart Disease'].value_counts().plot(kind='bar', color=['#2ecc71', '#e74c3c'], ax=ax)
ax.set_title('Distribuição da Variável Alvo (Heart Disease)', fontsize=14, fontweight='bold')
ax.set_xlabel('Heart Disease (0=Não, 1=Sim)')
ax.set_ylabel('Contagem')
ax.set_xticklabels(['Sem Doença (0)', 'Com Doença (1)'], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Codificação das variáveis categóricas com LabelEncoder
label_encoders = {}
df_encoded = df.copy()

for col in cat_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le
    print(f'\nCodificação de \'{col}\':')
    for original, encoded in zip(le.classes_, le.transform(le.classes_)):
        print(f'   {original} -> {encoded}')

In [ ]:
# Separação em X (features) e y (variável alvo)
X = df_encoded.drop('Heart Disease', axis=1)
y = df_encoded['Heart Disease']

print(f'Shape de X: {X.shape}')
print(f'Shape de y: {y.shape}')

In [ ]:
# Amostragem: divisão treino/teste (80% treino, 20% teste) com estratificação
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Divisão treino/teste (80/20) com estratificação:')
print(f'   Treino: {X_train.shape[0]} amostras')
print(f'   Teste:  {X_test.shape[0]} amostras')

In [ ]:
# Normalização dos dados com StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Dados normalizados com StandardScaler')

In [ ]:
# Matriz de correlação
fig, ax = plt.subplots(figsize=(14, 10))
correlation_matrix = df_encoded.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, ax=ax,
            square=True, linewidths=0.5)
ax.set_title('Matriz de Correlação das Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Treinamento dos Algoritmos

### 3.1 K-Nearest Neighbors (KNN)

In [ ]:
# Encontrar o melhor K usando validação cruzada
k_range = range(1, 31)
k_scores = []

for k in k_range:
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn_temp, X_train_scaled, y_train, cv=10, scoring='accuracy')
    k_scores.append(scores.mean())

best_k = list(k_range)[np.argmax(k_scores)]
print(f'Melhor K encontrado: {best_k} (Acurácia CV: {max(k_scores):.4f})')

In [ ]:
# Gráfico de seleção do melhor K
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_range, k_scores, 'o-', color='#3498db', linewidth=2, markersize=5)
ax.axvline(x=best_k, color='#e74c3c', linestyle='--', label=f'Melhor K = {best_k}')
ax.set_xlabel('Valor de K', fontsize=12)
ax.set_ylabel('Acurácia (Validação Cruzada)', fontsize=12)
ax.set_title('Seleção do Melhor K para KNN', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Treinar KNN com o melhor K
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)
y_prob_knn = knn.predict_proba(X_test_scaled)[:, 1]

print(f'KNN treinado com K={best_k}')

### 3.2 Regressão Logística

In [ ]:
# Treinar Regressão Logística
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

# Validação cruzada
lr_cv_scores = cross_val_score(lr, X_train_scaled, y_train, cv=10, scoring='accuracy')
print(f'Regressão Logística treinada')
print(f'Acurácia CV: {lr_cv_scores.mean():.4f} (+/- {lr_cv_scores.std():.4f})')

---
## 4. Métricas e Comparação dos Modelos

In [ ]:
def avaliar_modelo(nome, y_true, y_pred, y_prob):
    """Exibe todas as métricas de avaliação de um modelo."""
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)

    print(f'Métricas do modelo: {nome}')
    print(f'  Acurácia:    {acc:.4f}  ({acc*100:.2f}%)')
    print(f'  Precisão:    {prec:.4f}  ({prec*100:.2f}%)')
    print(f'  Recall:      {rec:.4f}  ({rec*100:.2f}%)')
    print(f'  F1-Score:    {f1:.4f}  ({f1*100:.2f}%)')
    print(f'  AUC-ROC:     {auc:.4f}  ({auc*100:.2f}%)')
    print(f'\nRelatório de Classificação:')
    print(classification_report(y_true, y_pred, target_names=['Sem Doença', 'Com Doença']))

    return {'Acurácia': acc, 'Precisão': prec, 'Recall': rec, 'F1-Score': f1, 'AUC-ROC': auc}

In [ ]:
# Avaliar KNN
metricas_knn = avaliar_modelo('KNN', y_test, y_pred_knn, y_prob_knn)

In [ ]:
# Avaliar Regressão Logística
metricas_lr = avaliar_modelo('Regressão Logística', y_test, y_pred_lr, y_prob_lr)

In [ ]:
# Tabela comparativa
comparacao = pd.DataFrame({
    'KNN': metricas_knn,
    'Regressão Logística': metricas_lr
})
print('TABELA COMPARATIVA DOS MODELOS')
print('=' * 50)
comparacao.round(4)

In [ ]:
# Indicar o melhor modelo
melhor_metrica = 'F1-Score'
if metricas_knn[melhor_metrica] > metricas_lr[melhor_metrica]:
    melhor_modelo = 'KNN'
    diferenca = metricas_knn[melhor_metrica] - metricas_lr[melhor_metrica]
else:
    melhor_modelo = 'Regressão Logística'
    diferenca = metricas_lr[melhor_metrica] - metricas_knn[melhor_metrica]

print(f'MELHOR MODELO (por {melhor_metrica}): {melhor_modelo}')
print(f'Diferença de {melhor_metrica}: {diferenca:.4f}')

In [ ]:
# Matrizes de confusão lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, titulo in [
    (axes[0], y_pred_knn, 'KNN'),
    (axes[1], y_pred_lr, 'Regressão Logística')
]:
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Sem Doença', 'Com Doença'],
                yticklabels=['Sem Doença', 'Com Doença'])
    ax.set_title(f'Matriz de Confusão - {titulo}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Real')
    ax.set_xlabel('Previsto')

plt.tight_layout()
plt.show()

In [ ]:
# Curvas ROC
fig, ax = plt.subplots(figsize=(8, 6))

fpr_knn, tpr_knn, _ = roc_curve(y_test, y_prob_knn)
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)

ax.plot(fpr_knn, tpr_knn, color='#3498db', linewidth=2,
        label=f'KNN (AUC = {metricas_knn["AUC-ROC"]:.4f})')
ax.plot(fpr_lr, tpr_lr, color='#e74c3c', linewidth=2,
        label=f'Reg. Logística (AUC = {metricas_lr["AUC-ROC"]:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aleatório (AUC = 0.5)')

ax.set_xlabel('Taxa de Falsos Positivos (FPR)', fontsize=12)
ax.set_ylabel('Taxa de Verdadeiros Positivos (TPR)', fontsize=12)
ax.set_title('Curvas ROC - Comparação dos Modelos', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico de barras comparativo
fig, ax = plt.subplots(figsize=(10, 6))
metricas_nomes = list(metricas_knn.keys())
valores_knn = list(metricas_knn.values())
valores_lr = list(metricas_lr.values())

x = np.arange(len(metricas_nomes))
width = 0.35

bars1 = ax.bar(x - width/2, valores_knn, width, label='KNN', color='#3498db', alpha=0.85)
bars2 = ax.bar(x + width/2, valores_lr, width, label='Regressão Logística', color='#e74c3c', alpha=0.85)

ax.set_ylim(0, 1.1)
ax.set_xlabel('Métricas', fontsize=12)
ax.set_ylabel('Valor', fontsize=12)
ax.set_title('Comparação de Métricas entre KNN e Regressão Logística', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metricas_nomes)
ax.legend(fontsize=11)

for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

---
## 5. Análise das Características Mais Difíceis de Aprender

### 5.1 Importância via Coeficientes da Regressão Logística
Features com **coeficientes absolutos menores** tiveram menor influência na decisão do modelo, indicando que foram mais difíceis de utilizar para a classificação.

In [ ]:
feature_names = X.columns.tolist()
coefs = np.abs(lr.coef_[0])
coefs_df = pd.DataFrame({
    'Feature': feature_names,
    'Coeficiente_Abs': coefs
}).sort_values('Coeficiente_Abs', ascending=True)

print('Features ordenadas por importância (menor = mais difícil de aprender):\n')
for _, row in coefs_df.iterrows():
    barra = '█' * int(row['Coeficiente_Abs'] * 30)
    print(f"   {row['Feature']:25s} | {row['Coeficiente_Abs']:.4f} | {barra}")

### 5.2 Impacto de Cada Feature na Acurácia (Remoção Sequencial)
Removemos cada feature individualmente e medimos o impacto na acurácia. Features cuja remoção **não reduz** (ou até melhora) a acurácia são as que os modelos têm mais dificuldade em aprender.

In [ ]:
baseline_acc_knn = accuracy_score(y_test, y_pred_knn)
baseline_acc_lr = accuracy_score(y_test, y_pred_lr)

resultados_remocao = []

for col in feature_names:
    cols_sem = [c for c in feature_names if c != col]
    X_train_sem = X_train[cols_sem]
    X_test_sem = X_test[cols_sem]

    scaler_temp = StandardScaler()
    X_train_sem_sc = scaler_temp.fit_transform(X_train_sem)
    X_test_sem_sc = scaler_temp.transform(X_test_sem)

    # KNN
    knn_temp = KNeighborsClassifier(n_neighbors=best_k)
    knn_temp.fit(X_train_sem_sc, y_train)
    acc_knn_sem = accuracy_score(y_test, knn_temp.predict(X_test_sem_sc))

    # Regressão Logística
    lr_temp = LogisticRegression(max_iter=1000, random_state=42)
    lr_temp.fit(X_train_sem_sc, y_train)
    acc_lr_sem = accuracy_score(y_test, lr_temp.predict(X_test_sem_sc))

    impacto_knn = baseline_acc_knn - acc_knn_sem
    impacto_lr = baseline_acc_lr - acc_lr_sem

    resultados_remocao.append({
        'Feature': col,
        'Impacto_KNN': impacto_knn,
        'Impacto_LR': impacto_lr,
        'Impacto_Medio': (impacto_knn + impacto_lr) / 2
    })

df_impacto = pd.DataFrame(resultados_remocao).sort_values('Impacto_Medio', ascending=True)
df_impacto.round(4)

In [ ]:
# Features mais difíceis de aprender
features_dificeis = df_impacto.head(5)['Feature'].tolist()

print('FEATURES MAIS DIFÍCEIS DE APRENDER')
print('=' * 55)
print('Quando remover uma feature NÃO reduz a acurácia (ou até melhora),')
print('significa que o modelo não conseguiu extrair padrões úteis dela.\n')

for i, feat in enumerate(features_dificeis, 1):
    row = df_impacto[df_impacto['Feature'] == feat].iloc[0]
    print(f'   {i}. {feat} (Impacto médio: {row["Impacto_Medio"]:.4f})')

In [ ]:
# Gráficos de importância das features
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Coeficientes da Regressão Logística
ax1 = axes[0]
colors = ['#e74c3c' if x < coefs_df['Coeficiente_Abs'].median() else '#3498db'
          for x in coefs_df['Coeficiente_Abs']]
ax1.barh(coefs_df['Feature'], coefs_df['Coeficiente_Abs'], color=colors)
ax1.set_xlabel('Coeficiente Absoluto', fontsize=11)
ax1.set_title('Importância (Reg. Logística)\n(Vermelho = Menor importância)', fontsize=12, fontweight='bold')

# Impacto na remoção
ax2 = axes[1]
colors2 = ['#e74c3c' if x <= 0 else '#2ecc71' for x in df_impacto['Impacto_Medio']]
ax2.barh(df_impacto['Feature'], df_impacto['Impacto_Medio'], color=colors2)
ax2.axvline(x=0, color='black', linewidth=0.8, linestyle='-')
ax2.set_xlabel('Impacto Médio na Acurácia', fontsize=11)
ax2.set_title('Impacto ao Remover Feature\n(Vermelho = Negativo/Neutro)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

---
## Conclusão

In [ ]:
print('=' * 60)
print('                   RESUMO FINAL')
print('=' * 60)
print(f'\nDataset: {df.shape[0]} amostras, {df.shape[1]} colunas')
print(f'Divisão: 80% treino / 20% teste (estratificada)')
print(f'Normalização: StandardScaler')
print(f'\n--- Resultados ---')
print(f'\nKNN (K={best_k}):')
for k, v in metricas_knn.items():
    print(f'   {k}: {v:.4f}')
print(f'\nRegressão Logística:')
for k, v in metricas_lr.items():
    print(f'   {k}: {v:.4f}')
print(f'\nMELHOR MODELO (por {melhor_metrica}): {melhor_modelo}')
print(f'\nFeatures mais difíceis de aprender:')
for i, f in enumerate(features_dificeis, 1):
    print(f'   {i}. {f}')
print('\n' + '=' * 60)